<a href="https://colab.research.google.com/github/nasif-raihan/GSU/blob/main/CSCI_7090_Module10_Lab1_SSDArchitecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — SSD Architecture (Lab)
**Computer Vision · Object Detection**

We build a **Single Shot Detector (SSD)** from a VGG16 base plus extra multi-scale layers, with parallel **localization** and **confidence** heads. Best run in Colab (downloads VGG16 weights). This mirrors the doc: base network → extra layers → multi-scale predictions.

## 1. Imports

In [1]:
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import VGG16_Weights

## 2. The SSD model
The VGG16 features up to conv5_3 form the base; five `extras` blocks add lower-resolution feature maps; `loc` and `conf` heads predict box offsets (4 per box) and class scores for **4 or 6** default boxes per cell.

In [2]:
class SSD(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.features = nn.ModuleList(vgg[:30])          # up to conv5_3

        self.extras = nn.ModuleList([
            nn.Sequential(nn.Conv2d(512, 1024, 3, padding=1), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(1024, 256, 1), nn.ReLU(inplace=True),
                          nn.Conv2d(256, 512, 3, stride=2, padding=1), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(512, 128, 1), nn.ReLU(inplace=True),
                          nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(256, 128, 1), nn.ReLU(inplace=True),
                          nn.Conv2d(128, 256, 3), nn.ReLU(inplace=True)),
            nn.Sequential(nn.Conv2d(256, 128, 1), nn.ReLU(inplace=True),
                          nn.Conv2d(128, 256, 3), nn.ReLU(inplace=True)),
        ])
        boxes = [4, 6, 6, 6, 4, 4]                        # default boxes per source
        chans = [512, 1024, 512, 256, 256, 256]
        self.loc  = nn.ModuleList([nn.Conv2d(ch, b*4, 3, padding=1) for ch, b in zip(chans, boxes)])
        self.conf = nn.ModuleList([nn.Conv2d(ch, b*num_classes, 3, padding=1) for ch, b in zip(chans, boxes)])

    def forward(self, x):
        locs, confs = [], []
        for layer in self.features:
            x = layer(x)
        locs.append(self.loc[0](x).permute(0, 2, 3, 1).contiguous())
        confs.append(self.conf[0](x).permute(0, 2, 3, 1).contiguous())
        for i, layer in enumerate(self.extras):
            x = layer(x)
            locs.append(self.loc[i+1](x).permute(0, 2, 3, 1).contiguous())
            confs.append(self.conf[i+1](x).permute(0, 2, 3, 1).contiguous())
        locs  = torch.cat([o.view(o.size(0), -1) for o in locs], 1)
        confs = torch.cat([o.view(o.size(0), -1) for o in confs], 1)
        return locs.view(locs.size(0), -1, 4), confs.view(confs.size(0), -1, self.num_classes)

## 3. Run a forward pass
The model returns localization `(N, num_boxes, 4)` and confidence `(N, num_boxes, num_classes)` — one prediction per default box across all feature-map scales.

In [3]:
num_classes = 21          # 20 VOC classes + background
ssd = SSD(num_classes)
x = torch.randn(1, 3, 300, 300)
locs, confs = ssd(x)
print("Localization predictions:", locs.size())
print("Confidence predictions:", confs.size())

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:10<00:00, 54.6MB/s]


Localization predictions: torch.Size([1, 3916, 4])
Confidence predictions: torch.Size([1, 3916, 21])


**What to notice:** the two heads share the same number of default boxes (first dim after batch). Each box gets **4** offset values and **num_classes** scores — the whole image is detected in a *single shot* across multiple scales.